In [1]:
import re
import zipfile
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

nltk.download('stopwords'); nltk.download('punkt'); nltk.download('punkt_tab')

# ===== EDA =====
# 1-2. Load (unzip first, then read the CSV inside)
df_imdb = pd.read_csv('IMDBDataset.csv')
df = df_imdb.iloc[:len(df_imdb) // 5].copy()   # use 1/5 to save time

# 3. Inspect
print(df.head())
print(df.shape)
print(df.dtypes)

# 4. NaN check
print(df.isna().sum())
df.dropna(inplace=True)

# 5. First 5 reviews + sentiment
print(df[['review', 'sentiment']].head())

# 6. Word count column
def count_words(text):
    return len(text.split())

df['words count'] = df['review'].apply(count_words)
print(df.head())

# ===== Preprocessing =====
stop = set(stopwords.words('english'))

# 1. simple_preprocessing
def simple_preprocessing(text):
    text = text.lower()
    text = re.sub(r'<br\s*/?>', ' ', text)              # html br tags
    text = re.sub(r'http\S+|www\.\S+', ' ', text)        # urls
    text = re.sub(r'[#@]', ' ', text)                    # hashtags/@ symbol
    text = re.sub(r'[^\w\s]', ' ', text)                 # punctuation
    tokens = word_tokenize(text)
    tokens = [w for w in tokens if w not in stop]        # stopwords
    return ' '.join(tokens)

# 2. Apply
df['review'] = df['review'].apply(simple_preprocessing)

# 3. Check
print(df['review'].head())

# 4. Duplicates
print('duplicates:', df.duplicated(subset='review').sum())
df.drop_duplicates(subset='review', inplace=True)
print('after drop:', df.duplicated(subset='review').sum())

# 5. Stemming
ps = PorterStemmer()
def stemming(text):
    return ' '.join(ps.stem(w) for w in text.split())

df['review'] = df['review'].apply(stemming)

# ===== Prepare data =====
# 1. X / Y  (binarize sentiment)
X = df['review']
y = df['sentiment'].map({'positive': 1, 'negative': 0})

# 2. TF-IDF
tfidf = TfidfVectorizer()
X = tfidf.fit_transform(X)

# 3. Split 30% test
x_train, x_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42)

# 4. Shapes
print(x_train.shape, x_test.shape)   # two values each
print(y_train.shape, y_test.shape)   # one value each

# ===== Model =====
model = LogisticRegression(max_iter=1000)   # 3-4. instantiate + train
model.fit(x_train, y_train)

# 5. Accuracy
y_pred = model.predict(x_test)
accuracy = accuracy_score(y_test, y_pred)
print('Accuracy:', accuracy)

# 7. Confusion matrix + report
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

# ===== Predict new reviews =====
new_reviews = ["I loved this movie!", "This movie was a bad comedy movie!"]
processed = [stemming(simple_preprocessing(r)) for r in new_reviews]
vec = tfidf.transform(processed)
preds = model.predict(vec)
label = {1: 'positive', 0: 'negative'}
for r, p in zip(new_reviews, preds):
    print(f'{r} -> {label[p]}')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
(10000, 2)
review       object
sentiment    object
dtype: object
review       0
sentiment    0
dtype: int64
                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
                                              review sentiment  words count
0  One of the other reviewers has mentioned that ...  positi